# Notebook 03 — Exploratory Data Analysis

> 9 visualisations covering seasons, teams, players, phases, and scoring trends.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
matches = pd.read_csv('../data/processed/matches_clean.csv')
deliveries = pd.read_csv('../data/processed/deliveries_clean.csv')
print('Data loaded.')

## EDA 1 — Matches Per Season

In [ ]:
mps = matches.groupby('season').size()
plt.figure(figsize=(12,4))
plt.bar(mps.index.astype(str), mps.values, color='steelblue', edgecolor='white')
plt.title('Matches per IPL Season (2008–2024)', fontsize=13, fontweight='bold')
plt.xlabel('Season'); plt.ylabel('Matches')
plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig('../reports/figures/fig01_matches_per_season.png', dpi=150); plt.show()
print(f'Range: {mps.min()} to {mps.max()} matches/season')

## EDA 2 — Toss Decision Trends

In [ ]:
toss_dec = matches['toss_decision'].value_counts()
fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].pie(toss_dec, labels=toss_dec.index, autopct='%1.1f%%', colors=['#4e79a7','#f28e2b'])
axes[0].set_title('Overall Toss Decision')
toss_by_season = matches.groupby(['season','toss_decision']).size().unstack(fill_value=0)
toss_by_season.plot(kind='bar', ax=axes[1], color=['#4e79a7','#f28e2b'])
axes[1].set_title('Field vs Bat by Season'); axes[1].set_xlabel('Season')
plt.tight_layout(); plt.savefig('../reports/figures/fig02_toss_decisions.png', dpi=150); plt.show()

## EDA 3 — Toss Advantage

In [ ]:
valid = matches[matches['winner'].notna()]
tw = valid.groupby('toss_decision')['toss_win_match'].mean() * 100
fig, ax = plt.subplots(figsize=(7,4))
tw.plot(kind='bar', ax=ax, color=['#59a14f','#e15759'])
ax.set_title('Win % for Toss Winner by Decision', fontsize=12, fontweight='bold')
ax.set_ylabel('Win %'); ax.set_ylim(40,60); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig('../reports/figures/fig03_toss_advantage.png', dpi=150); plt.show()
print(tw.round(2))

## EDA 4 — Team Total Wins

In [ ]:
wins = matches[matches['winner'].notna()]['winner'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(11,5))
wins.sort_values().plot(kind='barh', ax=ax, color='#4e79a7')
ax.set_title('Top 10 Teams — All-Time Wins', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figures/fig04_team_wins.png', dpi=150); plt.show()

## EDA 5 — Runs by Phase

In [ ]:
phase_runs = deliveries.groupby('phase')['total_runs'].mean()
phase_order = ['Powerplay','Middle','Death']
vals = [phase_runs[p] for p in phase_order]
fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(phase_order, vals, color=['#76b7b2','#edc948','#e15759'])
ax.set_title('Average Runs per Delivery by Phase', fontsize=12, fontweight='bold'); ax.set_ylabel('Avg Runs/Ball')
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center')
plt.tight_layout(); plt.savefig('../reports/figures/fig05_runs_by_phase.png', dpi=150); plt.show()

## EDA 6 — Top Batsmen

In [ ]:
top_bat = deliveries.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(11,6))
top_bat.sort_values().plot(kind='barh', ax=ax, color='#f28e2b')
ax.set_title('Top 15 Batsmen — Career IPL Runs', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figures/fig06_top_batsmen.png', dpi=150); plt.show()
print(f'Leader: {top_bat.index[0]} — {top_bat.iloc[0]} runs')

## EDA 7 — Top Bowlers

In [ ]:
wickets = deliveries[deliveries['is_wicket']==1]
top_bowl = wickets.groupby('bowler').size().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(11,6))
top_bowl.sort_values().plot(kind='barh', ax=ax, color='#59a14f')
ax.set_title('Top 15 Bowlers — Career IPL Wickets', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('../reports/figures/fig07_top_bowlers.png', dpi=150); plt.show()
print(f'Leader: {top_bowl.index[0]} — {top_bowl.iloc[0]} wickets')

## EDA 8 — Scoring Evolution

In [ ]:
match_scores = deliveries[deliveries['inning']==1].groupby('match_id')['total_runs'].sum().reset_index()
match_scores = match_scores.merge(matches[['id','season']], left_on='match_id', right_on='id')
avg_score = match_scores.groupby('season')['total_runs'].mean()
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(avg_score.index.astype(str), avg_score.values, marker='o', linewidth=2, color='#4e79a7')
ax.fill_between(range(len(avg_score)), avg_score.values, alpha=0.1, color='#4e79a7')
ax.set_title('Average 1st Innings Score by Season', fontsize=12, fontweight='bold')
plt.xticks(rotation=45); plt.tight_layout(); plt.savefig('../reports/figures/fig08_avg_score_trend.png', dpi=150); plt.show()

## EDA 9 — Dismissal Types

In [ ]:
dis = wickets[wickets['dismissal_kind']!='not_out']['dismissal_kind'].value_counts()
fig, ax = plt.subplots(figsize=(10,4))
dis.plot(kind='bar', ax=ax, color='#af7aa1')
ax.set_title('Dismissal Types in IPL', fontsize=12, fontweight='bold')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.savefig('../reports/figures/fig09_dismissal_types.png', dpi=150); plt.show()
print('Most common:', dis.index[0])